# Comparação corrigida das fronteiras Pareto 8D

Compara VRF-NBI, CNBI, NSGA-III e MOEA/D contra a referência completa. Todas as soluções são reavaliadas pelo mesmo RSM, sanitizadas e orientadas para minimização. GD1/IGD1 usam árvores k-d; HV usa Sobol QMC pareado com caixa validada. Pools são apenas diagnósticos.

Inferência: Wilcoxon pareado para NSGA-III × MOEA/D; Wilcoxon de uma amostra contra CNBI e VRF-NBI fixos; correção de Holm, efeito rank-biserial e IC bootstrap. CNBI × VRF-NBI é somente descritivo porque não possui replicações independentes.

In [ ]:
from pathlib import Path
import re, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import qmc, wilcoxon, rankdata
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from statsmodels.stats.multitest import multipletests
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
def root(start=Path.cwd()):
    for p in (start.resolve(),*start.resolve().parents):
        if (p/'AGENTS.md').exists(): return p
    raise FileNotFoundError('Raiz não encontrada')
ROOT=root(); HOME=Path.home(); ALPHA=2**0.75; OUT=ROOT/'results'/'applied'/'comparison_8d_corrected'; OUT.mkdir(parents=True,exist_ok=True)
EA_DIR=ROOT/'results'/'applied'/'applied_8d_equal_budget'/'final_fronts'
PSTAR=ROOT/'data'/'reference_fronts'/'referencia_pareto_8D_nsga3_moead.csv'
D=HOME/'Documents'/'Dissertação'/'04_CODIGOS'/'notebooks'/'files'; VRFP=HOME/'Documents'/'Dissertação'/'02_REFERENCIAS'/'Dados'/'VRF_Pareto.xlsx'; CNBIP=D/'fronteira_ND_cnbi.csv'; XP=HOME/'Documents'/'Dissertação'/'02_REFERENCIAS'/'Dados'/'VRF_artigo.xlsx'
XCOLS=['cs','f','md']; RESP=['T','MTTF','WR','Ra','Rt','Kp','ROI','OEE']; SIGNS=np.array([-1,-1,1,1,1,1,-1,-1.])
raw=pd.read_excel(XP); raw.columns=[str(c).strip() for c in raw.columns]; rsm=Pipeline([('poly',PolynomialFeatures(2,include_bias=False)),('reg',LinearRegression())]).fit(raw[XCOLS],raw[RESP])


In [ ]:
def nd(F): return NonDominatedSorting(method='efficient_non_dominated_sort').do(np.asarray(F,float),only_non_dominated_front=True)
def sanitize(name,X):
    X=np.asarray(X,float); n0=len(X); finite=np.isfinite(X).all(1); X=X[finite]; feasible=np.linalg.norm(X,axis=1)<=ALPHA+1e-8; ninv=int((~feasible).sum()); X=X[feasible]; _,i=np.unique(np.round(X,10),axis=0,return_index=True); X=X[np.sort(i)]; F=rsm.predict(X); i=nd(F*SIGNS); X,F=X[i],F[i]
    return {'name':name,'X':X,'F':F,'audit':{'source':name,'n_raw':n0,'n_nonfinite':int(n0-finite.sum()),'n_infeasible':ninv,'n_final_unique_ND':len(X)}}
P=sanitize('P* completa',pd.read_csv(PSTAR)[XCOLS]); V=sanitize('VRF-NBI',pd.read_excel(VRFP)[XCOLS]); C=sanitize('CNBI',pd.read_csv(CNBIP)[XCOLS])
def seed(p): return int(re.search(r'seed(\d+)',p.stem,re.I).group(1))
def load(prefix,label):
    out={}
    for p in sorted(EA_DIR.glob(f'{prefix}_seed*.csv')):
        d=pd.read_csv(p)
        if all(c in d for c in XCOLS): out[seed(p)]=sanitize(f'{label} seed {seed(p)}',d[XCOLS])
    return out
N=load('nsga3','NSGA-III'); M=load('moead','MOEA/D'); seeds=sorted(set(N)&set(M)); assert len(seeds)>=5
audit=pd.DataFrame([P['audit'],V['audit'],C['audit']]+[x['audit'] for x in N.values()]+[x['audit'] for x in M.values()]); audit.to_csv(OUT/'auditoria_sanitizacao.csv',index=False); display(audit)


In [ ]:
Pmin=P['F']*SIGNS; ideal=Pmin.min(0); amp=np.maximum(Pmin.max(0)-ideal,1e-12); norm=lambda F:(np.asarray(F)*SIGNS-ideal)/amp; Pn=norm(P['F']); tref=cKDTree(Pn)
def gd(A): return float(tref.query(A,k=1,workers=-1)[0].mean())
def igd(A): return float(cKDTree(A).query(Pn,k=1,workers=-1)[0].mean())
def near(A): return cKDTree(A).query(A,k=2,workers=-1)[0][:,1] if len(A)>1 else np.array([])
fronts=[Pn,norm(V['F']),norm(C['F'])]+[norm(x['F']) for x in N.values()]+[norm(x['F']) for x in M.values()]; lo=np.minimum(0,np.min(np.vstack([x.min(0) for x in fronts]),0)); mx=np.max(np.vstack([x.max(0) for x in fronts]),0); ref=np.maximum(1.1,mx+.05*np.maximum(mx-lo,1)); assert all(np.all(x<=ref+1e-12) for x in fronts)
def hv1(A,Z):
    A=A[nd(A)]; A=A[np.argsort(A.sum(1))]; hit=np.zeros(len(Z),bool)
    for zs in range(0,len(Z),2048):
        z=Z[zs:zs+2048]; h=np.zeros(len(z),bool)
        for ps in range(0,len(A),256):
            active=np.where(~h)[0]
            if not len(active): break
            h[active]=np.any(np.all(A[None,ps:ps+256]<=z[active,None],2),1)
        hit[zs:zs+len(z)]=h
    return float(hit.mean()*np.prod(ref-lo))
ZS=[]
for k in range(4):
    u=qmc.Sobol(8,scramble=True,seed=20260+k).random_base2(14); ZS.append(lo+u*(ref-lo))
def metrics(F):
    A=norm(F); d=near(A); h=np.array([hv1(A,z) for z in ZS]); return {'GD1':gd(A),'IGD1':igd(A),'HV':h.mean(),'HV_QMC_SE':h.std(ddof=1)/2,'Spacing':d.std(ddof=1) if len(d)>1 else np.nan,'Sparsity':d.mean() if len(d) else np.nan}


In [ ]:
det=pd.DataFrame([{'method':x['name'],'seed':np.nan,'n':len(x['F']),**metrics(x['F'])} for x in (V,C)])
rows=[]
for name,F in [('NSGA-III',N),('MOEA/D',M)]:
    for s in seeds: rows.append({'method':name,'seed':s,'n':len(F[s]['F']),**metrics(F[s]['F'])})
per=pd.DataFrame(rows); summary=per.groupby('method').agg(n_seeds=('seed','size'),n_median=('n','median'),GD1_median=('GD1','median'),GD1_IQR=('GD1',lambda x:x.quantile(.75)-x.quantile(.25)),IGD1_median=('IGD1','median'),IGD1_IQR=('IGD1',lambda x:x.quantile(.75)-x.quantile(.25)),HV_median=('HV','median'),HV_IQR=('HV',lambda x:x.quantile(.75)-x.quantile(.25)),Spacing_median=('Spacing','median'),Sparsity_median=('Sparsity','median')).reset_index()
pools=[]
for name,F in [('NSGA-III',N),('MOEA/D',M)]:
    A=np.vstack([F[s]['F'] for s in seeds]); A=A[nd(A*SIGNS)]; pools.append({'method':name+' pool diagnóstico','n':len(A),**metrics(A)})
pools=pd.DataFrame(pools); det.to_csv(OUT/'metricas_deterministicos.csv',index=False); per.to_csv(OUT/'metricas_por_seed.csv',index=False); summary.to_csv(OUT/'resumo_eas.csv',index=False); pools.to_csv(OUT/'pools_diagnosticos.csv',index=False); display(det); display(summary)


In [ ]:
METS=['GD1','IGD1','HV','Spacing','Sparsity']; LOWER={'GD1','IGD1','Spacing','Sparsity'}
def rbc(d):
    d=np.asarray(d); d=d[np.isfinite(d)&(d!=0)]
    if not len(d): return 0.
    r=rankdata(abs(d)); return float((r[d>0].sum()-r[d<0].sum())/r.sum())
tests=[]
for met in METS:
    a=per[per.method=='NSGA-III'].set_index('seed').loc[seeds,met].to_numpy(); b=per[per.method=='MOEA/D'].set_index('seed').loc[seeds,met].to_numpy(); fav=b-a if met in LOWER else a-b; w=wilcoxon(fav); tests.append(['NSGA-III vs MOEA/D',met,'Wilcoxon pareado',len(a),w.statistic,w.pvalue,rbc(fav),np.median(fav)])
    for name,x in [('NSGA-III',a),('MOEA/D',b)]:
        for base in ('CNBI','VRF-NBI'):
            v=float(det.loc[det.method==base,met].iloc[0]); fav=v-x if met in LOWER else x-v; w=wilcoxon(fav); tests.append([f'{name} vs {base}',met,'Wilcoxon uma amostra',len(x),w.statistic,w.pvalue,rbc(fav),np.median(fav)])
tests=pd.DataFrame(tests,columns=['comparison','metric','test','n','statistic','p_raw','rank_biserial_positive_favors_first','median_favorable_difference']); tests['p_holm']=multipletests(tests.p_raw,method='holm')[1]; tests['significant_holm_0p05']=tests.p_holm<.05
rng=np.random.default_rng(9876); ci=[]
for name in ('NSGA-III','MOEA/D'):
    for met in METS:
        x=per.loc[per.method==name,met].to_numpy(); bs=np.median(rng.choice(x,(20000,len(x)),replace=True),1); q=np.percentile(bs,[2.5,97.5]); ci.append([name,met,np.median(x),q[0],q[1],len(x)])
ci=pd.DataFrame(ci,columns=['method','metric','median','bootstrap_CI95_low','bootstrap_CI95_high','n']); tests.to_csv(OUT/'testes_wilcoxon_holm.csv',index=False); ci.to_csv(OUT/'bootstrap_IC95_mediana.csv',index=False); display(tests); display(ci)


In [ ]:
# Redução equalizada: nomes corretos, cardinalidade garantida e seleção final pela mediana em cinco sementes.
F=norm(C['F']); target=len(V['F'])
def fps(A,n,s):
    rng=np.random.default_rng(s); out=[int(rng.integers(len(A)))]; d=np.linalg.norm(A-A[out[0]],1) if False else np.linalg.norm(A-A[out[0]],axis=1)
    while len(out)<n:
        j=int(np.argmax(d)); out.append(j); d=np.minimum(d,np.linalg.norm(A-A[j],axis=1))
    return np.array(out)
def reps(A,n,model,s):
    lab=model.fit_predict(A); cen=model.cluster_centers_ if hasattr(model,'cluster_centers_') else model.means_; out=[]
    for k in range(n):
        z=np.where(lab==k)[0]
        if len(z): out.append(z[np.argmin(np.linalg.norm(A[z]-cen[k],axis=1))])
    out=list(dict.fromkeys(out)); rem=np.setdiff1d(np.arange(len(A)),out)
    if len(out)<n: out.extend(rem[fps(A[rem],n-len(out),s)])
    assert len(set(out))==n; return np.array(out)
def hier(A,n):
    lab=AgglomerativeClustering(n_clusters=n,linkage='ward').fit_predict(A); return np.array([z[np.argmin(np.linalg.norm(A[z]-A[z].mean(0),axis=1))] for k in range(n) for z in [np.where(lab==k)[0]]])
def dirs(A,n,s):
    W=np.random.default_rng(s).dirichlet(np.ones(8),n); W/=np.linalg.norm(W,axis=1,keepdims=True); B=np.maximum(A,0); U=B/np.maximum(np.linalg.norm(B,axis=1,keepdims=True),1e-15); lab=np.argmax(U@W.T,1); out=[]
    for k in range(n):
        z=np.where(lab==k)[0]
        if len(z): out.append(z[np.argmin(np.linalg.norm(B[z],axis=1))])
    out=list(dict.fromkeys(out)); rem=np.setdiff1d(np.arange(len(A)),out)
    if len(out)<n: out.extend(rem[fps(A[rem],n-len(out),s)])
    return np.array(out)
rr=[]
for s in range(1,6):
    methods={'Farthest-point':fps(F,target,s),'K-Means + representante real':reps(F,target,KMeans(target,random_state=s,n_init=10),s),'MiniBatchKMeans + representante real':reps(F,target,MiniBatchKMeans(target,random_state=s,n_init=10,batch_size=256),s),'GMM + representante real':reps(F,target,GaussianMixture(target,random_state=s,n_init=3,covariance_type='diag'),s),'Direções + convergência no nicho':dirs(F,target,s),'Hierárquica Ward':hier(F,target)}
    for name,i in methods.items(): assert len(np.unique(i))==target; rr.append({'method':name,'seed':s,'n':len(i),**metrics(C['F'][i])})
red=pd.DataFrame(rr); redsum=red.groupby('method').agg(IGD1_median=('IGD1','median'),IGD1_IQR=('IGD1',lambda x:x.quantile(.75)-x.quantile(.25)),GD1_median=('GD1','median'),HV_median=('HV','median'),Spacing_median=('Spacing','median'),Sparsity_median=('Sparsity','median')).sort_values('IGD1_median').reset_index(); red.to_csv(OUT/'reducao_por_seed.csv',index=False); redsum.to_csv(OUT/'reducao_resumo.csv',index=False); display(redsum)


In [ ]:
fig,ax=plt.subplots(1,3,figsize=(15,5))
for a,met in zip(ax,['GD1','IGD1','HV']):
    a.boxplot([per.loc[per.method==m,met] for m in ('NSGA-III','MOEA/D')],tick_labels=['NSGA-III','MOEA/D'])
    for _,r in det.iterrows(): a.axhline(r[met],ls='--',label=r.method)
    a.set_title(met); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT/'metricas_principais.png',dpi=180); plt.show()
with pd.ExcelWriter(OUT/'comparacao_fronteiras_8D_corrigida.xlsx',engine='openpyxl') as w:
    audit.to_excel(w,'auditoria',index=False); det.to_excel(w,'deterministicos',index=False); per.to_excel(w,'EAs_por_seed',index=False); summary.to_excel(w,'resumo_EAs',index=False); pools.to_excel(w,'pools_diagnosticos',index=False); tests.to_excel(w,'testes_Holm',index=False); ci.to_excel(w,'bootstrap_IC95',index=False); red.to_excel(w,'reducao_por_seed',index=False); redsum.to_excel(w,'reducao_resumo',index=False)
manifest={'Pstar':str(PSTAR),'Pstar_sha256':hashlib.sha256(PSTAR.read_bytes()).hexdigest(),'paired_seeds':seeds,'GD_IGD':'p=1 euclidiana','HV':{'scrambles':4,'points_each':2**14,'lower':lo.tolist(),'reference':ref.tolist()},'multiplicity':'Holm'}; (OUT/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8'); print(OUT)
